# Sudoku-07 : Résolution par Propagation de Contraintes (Norvig)

**Navigation** : [<< Sudoku-06 AIMA-CSP](Sudoku-06-AIMA-CSP-CSharp.ipynb) | [Index](README.md) | [Sudoku-08 HumanStrategies >>](Sudoku-08-HumanStrategies-CSharp.ipynb)

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. Comprendre l'approche de Peter Norvig pour la résolution de Sudoku par propagation de contraintes
2. Implémenter les stratégies d'élimination et de hidden singles (seul emplacement)
3. Combiner propagation de contraintes et recherche récursive avec heuristique MRV
4. Comparer les performances avec les solveurs précédents (Backtracking, OR-Tools)

### Prérequis
- [Sudoku-0-Environment](Sudoku-00-Environment-CSharp.ipynb) : classes `SudokuGrid`, `ISudokuSolver`, `SudokuHelper`
- Notions de base en résolution de problèmes de satisfaction de contraintes (CSP)

### Durée estimée : 20 minutes

### Liens
- Voir [CSP-2-Consistance](../Search/Part2-CSP/CSP-2-Consistency.ipynb) pour la théorie de la propagation de contraintes
- [Article original de Peter Norvig (2006)](http://norvig.com/sudoku.html)


## 1. Introduction

En 2006, Peter Norvig a publié un article devenu célèbre dans lequel il présente un solveur de Sudoku remarquablement concis et performant. La thèse défendue dans l'article est la suivante : **la propagation de contraintes suffit à résoudre la plupart des grilles de Sudoku sans aucune recherche** — ce carnet l'illustre sur la grille facile de référence, refermée par pure propagation en section 3, sans l'établir sur un corpus : la mesure de la fraction refermée par propagation seule sur les 157 grilles du benchmark est l'exemple guidé à compléter (section 6).

Le solveur repose sur deux stratégies de propagation :

| Stratégie | Description | Équivalent CSP |
|-----------|-------------|----------------|
| **Élimination** | Si une cellule a une valeur assignée, retirer cette valeur de tous ses voisins (même ligne, colonne, boite) | Arc-consistency |
| **Hidden single** (seul emplacement) | Si dans une unité (ligne, colonne ou boite), une valeur n'a qu'un seul emplacement possible, l'y assigner | Hidden single |

Lorsque la propagation seule ne suffit pas (typiquement sur les grilles difficiles), un mécanisme de **recherche récursive avec backtracking** prend le relais. L'heuristique **MRV** (Minimum Remaining Values) guide le choix de la prochaine cellule à explorer : on choisit celle qui a le moins de candidats, ce qui réduit l'arbre de recherche.

C'est le fil des sections : structures de données et vérification de leur géométrie (2), propagation et ses limites mesurées (3), recherche complémentaire guidée par MRV (4), benchmark face à un témoin backtracking (5), puis deux exemples guidés (6) — dont le second, à compléter, mesure exactement la fraction du corpus que la propagation referme seule.

### Importation des classes de base

Nous importons les classes définies dans le notebook d'environnement.

In [1]:
#!import Sudoku-00-Environment-CSharp.ipynb

# Sudoku-00 : Environnement et Classes de Base (C#)

**Navigation** : [Index](README.md) | [Sudoku-01 Backtracking C# >>](Sudoku-01-Backtracking-CSharp.ipynb)

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. Comprendre la structure de données `SudokuGrid` et ses méthodes principales
2. Utiliser `ISudokuSolver` pour implémenter un solveur de Sudoku
3. Exploiter `SudokuHelper` pour charger des grilles et tester des solveurs
4. Comparer les performances de plusieurs solveurs sur différentes difficultés

**Prérequis** : Notions de base en C# (.NET Interactive)  
**Durée estimée** : ~15 min

Installing Packages Plotly.NET

## Définition de la classe SudokuGrid

Nous définissons ici la classe SudokuGrid qui représente une grille de Sudoku et fournit des méthodes pour manipuler et afficher les grilles.


SudokuGrid defini.


### Interprétation : Structure de données pour la grille Sudoku

**Sortie obtenue** : La classe `SudokuGrid` encapsule toutes les opérations de manipulation, validation et affichage d'une grille de Sudoku 9x9.

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| `Cells[9,9]` | int[,] | Stockage interne des valeurs (0 = vide) |
| `AllNeighbours` | 27 x 9 positions | Pré-calcul des voisins ligne/colonne/bloc |
| `CellNeighbours[9][9]` | ~20 positions chacune | Voisins directs de chaque cellule |
| `GetAvailableNumbers()` | int[] | Candidats valides pour une cellule |
| `NbErrors()` | int | Nombre de conflits + modifications erronées |

**Points clés** :
1. **Pré-calcul des voisins** : `AllNeighbours` et `CellNeighbours` sont calculés une seule fois à l'initialisation, évitant les recalculs coûteux
2. **Conversion flexible** : Méthodes pour convertir entre tableaux 1D, 2D et jagged arrays (utile pour différents formats de fichiers)
3. **Validation robuste** : `NbErrors` compte à la fois les doublons (ligne/colonne/bloc) et les modifications de indices pré-remplis
4. **Parsing tolerant** : `ReadMultiSudoku` accepte plusieurs formats (`.`, `X`, `-`, espaces)

> **Note technique** : La structure `CellNeighbours[i][j]` contient environ 20 positions (8 ligne + 8 colonne + 4 bloc, moins les doublons). Ce pré-calcul est crucial pour les performances des algorithmes de backtracking et de propagation de contraintes.

## Définition de l'interface ISudokuSolver

Nous définissons ici l'interface ISudokuSolver qui sera implémentée par les différentes stratégies de résolution de Sudoku.


... (4 outputs left out)

### Interprétation : Infrastructure de test et benchmark

**Sortie obtenue** : La classe `SudokuHelper` fournit une infrastructure complète pour tester et comparer les solveurs de Sudoku.

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| `GetSudokus()` | 51/95/100 grilles | Trois niveaux de difficulté (Easy/Medium/Hard) |
| `TestSolvers()` | Performance multi-solveurs | Exécution parallèle avec timeout |
| `DisplayResults()` | Graphiques SVG inline (SvgChartHelper) | Comparaison des temps par difficulté, sérialisée dans le notebook |
| `SolveSudoku()` | Test unitaire | Résolution individuelle avec affichage |

**Points clés** :
1. **Chargement intelligent** : Recherche récursive du dossier `Puzzles` dans l'arborescence
2. **Robustesse** : Gestion des timeouts (3 000 ms par défaut — paramètre de configuration du solveur, valeur fixée dans le code) et exceptions
3. **Mesures** : Temps d'exécution total + nombre de grilles resolues
4. **Disqualification** : Un solver échouant sur une grille est disqualifié pour la difficulté

> **Note technique** : La méthode `TestSolvers` utilise `Interlocked.Increment` pour un thread-safe incrément du compteur de solutions. Le `CancellationToken` permet d'interrompre proprement les solveurs trop lents.

## Exercice : Validation d'une grille Sudoku

### Énoncé

Implémentez une méthode `IsValidSolution` qui vérifie qu'une grille est une solution valide de Sudoku, c'est-à-dire que chaque ligne, chaque colonne et chaque bloc 3x3 contient exactement une fois chaque chiffre de 1 à 9.

Utilisez cette méthode pour valider les résultats de `SudokuHelper.SolveSudoku`.

**Indices :**

- Parcourez les 9 lignes, 9 colonnes et 9 blocs
- Pour chaque unité, verifiez que les 9 chiffres sont tous présents sans doublon
- `SudokuGrid.AllNeighbours` contient déjà les indices des unités

Exercice a completer


## Résumé et perspectives

Ce notebook a posé les fondations de toute la série Sudoku en définissant trois composants essentiels. La classe `SudokuGrid` encapsule la représentation d'une grille 9x9 avec le pré-calcul des voisins (`AllNeighbours`, `CellNeighbours`), ce qui évite les recalculs coûteux lors de la résolution. L'interface `ISudokuSolver` implante le pattern Stratégie, permettant de permuter les algorithmes de résolution sans modifier le code client. Enfin, la classe `SudokuHelper` fournit une infrastructure de benchmark complète avec chargement de puzzles, mesures de performance et visualisation SVG inline (SvgChartHelper, zéro dépendance).

L'infrastructure de test (`TestSolvers`, `DisplayResults`) permet de comparer objectivement les solveurs sur trois niveaux de difficulté (Easy, Medium, Hard) avec gestion des timeouts et des disqualifications. Ce cadre de benchmark sera utilisé dans tous les notebooks suivants pour mesurer les performances de chaque algorithme.

Le notebook suivant, [Sudoku-01-Backtracking](Sudoku-01-Backtracking-CSharp.ipynb), utilise ces classes pour implémenter le premier algorithme de résolution : le backtracking récursif avec ses heuristiques d'amélioration.

### Vérification de l'environnement et premier contact avec les grilles

La cellule suivante charge les classes de `Sudoku-00-Environment` via la directive `#!import` : `SudokuGrid`, `ISudokuSolver`, `SudokuHelper` et les stubs d'exercices — la sortie confirme les définitions. Elle affiche ensuite la première grille de chaque difficulté. Cet affichage n'est pas une formalité : ces trois grilles **exactes** sont les cobayes de tout le notebook — propagation seule en section 3, solveur complet en section 4, benchmark en section 5 — et ce sont les mêmes fichiers que ceux du Sudoku-01 (la comparaison finale de la section 5 se fera donc à grille identique). Observez déjà la densité décroissante des indices, de la grille facile presque pleine à la grille difficile clairsemée : c'est la seule différence visible entre ce que les deux solveurs devront affronter.

In [2]:
// Affichage d'un puzzle par difficulte
var easySudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Easy).First();
display($"Puzzle Facile :\n{easySudoku}");

var mediumSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Medium).First();
display($"Puzzle Moyen :\n{mediumSudoku}");

var hardSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Hard).First();
display($"Puzzle Difficile :\n{hardSudoku}");

Puzzle Facile :
-------------------------------
| 9     2 |       5 | 4     3 | 
| 1       |    6  3 |    2  5 | 
| 5     8 | 4     7 |    6    | 
-------------------------------
|    2  6 | 3     9 |       1 | 
|    5  7 |    1    | 2  9    | 
|    9    | 6  7    | 5  3    | 
-------------------------------
| 2  4    | 5  3    | 6       | 
| 7     5 | 2       | 3     4 | 
|    8    |    4  1 | 9  5    | 
-------------------------------

Puzzle Moyen :
-------------------------------
| 8  5    |       2 | 4       | 
| 7  2    |         |       9 | 
|       4 |         |         | 
-------------------------------
|         | 1     7 |       2 | 
| 3     5 |         | 9       | 
|    4    |         |         | 
-------------------------------
|         |    8    |    7    | 
|    1  7 |         |         | 
|         |    3  6 |    4    | 
-------------------------------

Puzzle Difficile :
-------------------------------
| 4       |         | 8     5 | 
|    3    |         |         | 
|         | 7       |         | 
-------------------------------
|    2    |         |    6    | 
|         |    8    | 4       | 
|         |    1    |         | 
-------------------------------
|         | 6     3 |    7    | 
| 5       | 2       |         | 
| 1     4 |         |         | 
-------------------------------

### Lecture des trois grilles : 45, 22 puis 17 indices

Comptez les cases remplies de chaque grille : la **facile** porte 45 indices (5 par ligne), la **moyenne** 22, la **difficile** 17. La gradation est frappante : la grille difficile a moins de la moitié des indices de la facile — 64 cellules à déduire au lieu de 36. Retenez deux choses de cet affichage. D'abord, la difficulté d'un Sudoku ne se lit pas seulement au **nombre** d'indices mais à leur **placement** (deux grilles à 17 indices peuvent différer énormément selon les interactions). Ensuite, ces trois grilles sont celles du benchmark de la section 5 : les temps comparés plus bas le seront sur ces inputs exacts — l'affichage présent fixe le référentiel visuel de ce qui suivra.

## 2. Structures de données

L'approche de Norvig repose sur une structure de données centrale : pour chaque cellule de la grille, on maintient l'ensemble des valeurs encore possibles.

| Structure | Type | Description |
|-----------|------|-------------|
| `_possible` | `Dictionary<(int,int), HashSet<int>>` | Valeurs candidates pour chaque cellule |
| `_units` | `(int,int)[][]` | Les 27 unités (rangées, colonnes et boites) |
| `_peers` | `Dictionary<(int,int), HashSet<(int,int)>>` | Voisins de chaque cellule (20 cellules par cellule) |

**Initialisation** :
- Chaque cellule vide demarre avec les candidats {1, 2, ..., 9}
- Chaque cellule pré-remplie demarre avec {valeur_donnee}
- Les voisins (peers) d'une cellule sont toutes les cellules partageant sa ligne, sa colonne ou sa boite, soit 20 cellules

Nous allons d'abord définir les structures statiques (unités et voisins) qui ne dependent pas de la grille.


## Exercice : Compter les singletons caches (Hidden Singles)

**Objectif :**
Implementez la detection des Hidden Singles : une valeur qui ne peut aller
que dans une seule cellule d'une unité (ligne/colonne/bloc).

**Indice :**
Pour chaque valeur 1-9, comptez dans combien de cellules d'une unité elle peut apparaitre.
Si exactement 1, c'est un Hidden Single.


In [3]:
// EXERCICE : Compter les singletons caches (Hidden Singles)
public List<(int Row, int Col, int Value)> FindHiddenSingles(Dictionary<(int, int), HashSet<int>> candidates)
{
    // TODO: Parcourez chaque unite et trouvez les valeurs qui ne peuvent
    // aller que dans une seule cellule de cette unite
    return null; // TODO etudiant
}
Console.WriteLine("Exercice a completer");

Exercice a completer


In [4]:
// Structures statiques partagees par toutes les instances du solveur

// Les 27 unites : 9 lignes + 9 colonnes + 9 boites
// Chaque unite est un tableau de 9 positions (row, col)
static (int row, int col)[][] BuildUnits()
{
    var units = new List<(int, int)[]>();

    // 9 lignes
    for (int r = 0; r < 9; r++)
        units.Add(Enumerable.Range(0, 9).Select(c => (r, c)).ToArray());

    // 9 colonnes
    for (int c = 0; c < 9; c++)
        units.Add(Enumerable.Range(0, 9).Select(r => (r, c)).ToArray());

    // 9 boites 3x3
    for (int br = 0; br < 3; br++)
        for (int bc = 0; bc < 3; bc++)
            units.Add(
                Enumerable.Range(0, 3).SelectMany(r =>
                    Enumerable.Range(0, 3).Select(c => (br * 3 + r, bc * 3 + c))
                ).ToArray()
            );

    return units.ToArray();
}

// Pour chaque cellule, la liste des unites auxquelles elle appartient
static Dictionary<(int, int), List<(int, int)[]>> BuildCellUnits((int, int)[][] allUnits)
{
    var cellUnits = new Dictionary<(int, int), List<(int, int)[]>>();
    for (int r = 0; r < 9; r++)
        for (int c = 0; c < 9; c++)
            cellUnits[(r, c)] = allUnits.Where(u => u.Contains((r, c))).ToList();
    return cellUnits;
}

// Pour chaque cellule, l'ensemble de ses voisins (peers)
static Dictionary<(int, int), HashSet<(int, int)>> BuildPeers(
    Dictionary<(int, int), List<(int, int)[]>> cellUnits)
{
    var peers = new Dictionary<(int, int), HashSet<(int, int)>>();
    for (int r = 0; r < 9; r++)
        for (int c = 0; c < 9; c++)
        {
            var cell = (r, c);
            peers[cell] = new HashSet<(int, int)>(
                cellUnits[cell].SelectMany(u => u).Where(p => p != cell)
            );
        }
    return peers;
}

var allUnits = BuildUnits();
var cellUnits = BuildCellUnits(allUnits);
var peers = BuildPeers(cellUnits);

// Verification
display($"Nombre d'unites : {allUnits.Length} (attendu : 27)");
display($"Nombre de voisins de la cellule (0,0) : {peers[(0,0)].Count} (attendu : 20)");

Nombre d'unites : 27 (attendu : 27)

Nombre de voisins de la cellule (0,0) : 20 (attendu : 20)

### Interpretation : structures statiques

Les structures calculees ci-dessus sont independantes de la grille à résoudre :

| Vérification | Valeur attendue | Explication |
|--------------|-----------------|-------------|
| Nombre d'unités | 27 | 9 lignes + 9 colonnes + 9 boites |
| Voisins par cellule | 20 | 8 (ligne) + 8 (colonne) + 8 (boite) - 4 (comptes deux fois) |

Les deux lignes imprimées portent exactement ces valeurs attendues — `Nombre d'unites : 27 (attendu : 27)` et `Nombre de voisins de la cellule (0,0) : 20 (attendu : 20)` — le code croise ainsi sa géométrie avec les invariants théoriques avant même qu'aucune grille n'entre en jeu. C'est sur ces fondations vérifiées, construites une seule fois et partagées entre toutes les instances, que la section suivante bâtit le moteur de propagation.

## 3. Propagation de contraintes

La propagation de contraintes est le coeur de l'algorithme de Norvig. Elle repose sur deux règles appliquees en boucle jusqu'a stabilisation :

### Règle 1 : Élimination

> Si une cellule `(r, c)` n'a plus qu'un seul candidat `v`, alors `v` doit etre retire des candidats de tous les voisins de `(r, c)`.

C'est l'équivalent de la propagation d'arc-consistance dans la théorie des CSP.

### Règle 2 : Hidden single (seul emplacement)

> Si dans une unité (ligne, colonne ou boite), une valeur `v` n'apparaît comme candidat que dans une seule cellule, alors `v` doit etre assigne a cette cellule.

Ces deux règles se renforcent mutuellement : l'élimination peut créer des naked singles, et l'assignation d'un hidden single déclenche de nouvelles eliminations.

**Gestion des contradictions** : si l'élimination retire le dernier candidat d'une cellule, on a détecté une contradiction. La méthode renvoie `false` pour signaler l'echec.

Implementons d'abord les méthodes `Eliminate` et `Assign`, puis la boucle de propagation.


In [5]:
/// <summary>
/// Moteur de propagation de contraintes a la Norvig.
/// Maintient un dictionnaire de candidats par cellule et applique
/// les règles d'élimination et de naked single en boucle.
/// </summary>
public class NorvigPropagation
{
    // Structures statiques (partagees entre instances)
    private static readonly (int, int)[][] Units;
    private static readonly Dictionary<(int, int), List<(int, int)[]>> CellUnits;
    private static readonly Dictionary<(int, int), HashSet<(int, int)>> Peers;

    static NorvigPropagation()
    {
        Units = BuildUnitsStatic();
        CellUnits = BuildCellUnitsStatic(Units);
        Peers = BuildPeersStatic(CellUnits);
    }

    // Candidats pour chaque cellule
    private Dictionary<(int, int), HashSet<int>> _possible;

    // Compteur de propagations pour diagnostic
    public int PropagationCount { get; private set; }

    /// <summary>
    /// Initialise les candidats a partir d'une grille et lance la propagation initiale.
    /// Renvoie true si la grille est coherente, false si une contradiction est detectee.
    /// </summary>
    public bool Initialize(SudokuGrid grid)
    {
        PropagationCount = 0;
        _possible = new Dictionary<(int, int), HashSet<int>>();

        // Toutes les cellules demarrent avec {1..9}
        for (int r = 0; r < 9; r++)
            for (int c = 0; c < 9; c++)
                _possible[(r, c)] = new HashSet<int> { 1, 2, 3, 4, 5, 6, 7, 8, 9 };

        // Assigner les valeurs données (ce qui déclenche la propagation)
        for (int r = 0; r < 9; r++)
            for (int c = 0; c < 9; c++)
                if (grid.Cells[r, c] > 0)
                    if (!Assign(r, c, grid.Cells[r, c]))
                        return false; // Contradiction detectee

        return true;
    }

    /// <summary>
    /// Assigne la valeur v a la cellule (row, col) en eliminant toutes les autres valeurs.
    /// </summary>
    public bool Assign(int row, int col, int value)
    {
        // Éliminer toutes les valeurs sauf 'value'
        var otherValues = _possible[(row, col)].Where(v => v != value).ToList();
        foreach (var other in otherValues)
            if (!Eliminate(row, col, other))
                return false;
        return true;
    }

    /// <summary>
    /// Retire la valeur v des candidats de la cellule (row, col).
    /// Applique les deux règles de propagation si nécessaire.
    /// </summary>
    public bool Eliminate(int row, int col, int value)
    {
        var cell = (row, col);

        // Si la valeur n'est deja plus candidate, rien a faire
        if (!_possible[cell].Contains(value))
            return true;

        _possible[cell].Remove(value);
        PropagationCount++;

        // Regle 1 : si la cellule n'a plus de candidat, contradiction
        if (_possible[cell].Count == 0)
            return false;

        // Regle 1 (suite) : si la cellule n'a plus qu'un candidat,
        // l'éliminer de tous les voisins
        if (_possible[cell].Count == 1)
        {
            int remaining = _possible[cell].First();
            foreach (var peer in Peers[cell])
                if (!Eliminate(peer.Item1, peer.Item2, remaining))
                    return false;
        }

        // Regle 2 : pour chaque unite contenant cette cellule,
        // vérifier si 'value' n'a plus qu'un seul emplacement possible
        foreach (var unit in CellUnits[cell])
        {
            var placesForValue = unit.Where(pos => _possible[pos].Contains(value)).ToList();

            if (placesForValue.Count == 0)
                return false; // Aucun emplacement pour cette valeur dans l'unite

            if (placesForValue.Count == 1)
            {
                // Naked single : assigner la valeur a la seule cellule possible
                var target = placesForValue[0];
                if (!Assign(target.Item1, target.Item2, value))
                    return false;
            }
        }

        return true;
    }

    /// <summary>
    /// Verifie si la grille est resolue (chaque cellule a exactement un candidat).
    /// </summary>
    public bool IsSolved()
    {
        return _possible.Values.All(candidates => candidates.Count == 1);
    }

    /// <summary>
    /// Renvoie les candidats d'une cellule.
    /// </summary>
    public HashSet<int> GetCandidates(int row, int col) => _possible[(row, col)];

    /// <summary>
    /// Cree une copie profonde de l'etat des candidats (pour le backtracking).
    /// </summary>
    public Dictionary<(int, int), HashSet<int>> CloneState()
    {
        return _possible.ToDictionary(
            kvp => kvp.Key,
            kvp => new HashSet<int>(kvp.Value)
        );
    }

    /// <summary>
    /// Restaure l'etat des candidats a partir d'une copie.
    /// </summary>
    public void RestoreState(Dictionary<(int, int), HashSet<int>> state)
    {
        _possible = state;
    }

    /// <summary>
    /// Renvoie la cellule non resolue ayant le moins de candidats (heuristique MRV).
    /// </summary>
    public (int row, int col)? GetMrvCell()
    {
        (int, int)? best = null;
        int bestCount = int.MaxValue;

        foreach (var kvp in _possible)
        {
            if (kvp.Value.Count > 1 && kvp.Value.Count < bestCount)
            {
                bestCount = kvp.Value.Count;
                best = kvp.Key;
            }
        }

        return best;
    }

    /// <summary>
    /// Ecrit les valeurs resolues dans une SudokuGrid.
    /// </summary>
    public void WriteTo(SudokuGrid grid)
    {
        foreach (var kvp in _possible)
        {
            if (kvp.Value.Count == 1)
                grid.Cells[kvp.Key.Item1, kvp.Key.Item2] = kvp.Value.First();
        }
    }

    // --- Methodes statiques de construction (identiques a celles definies plus haut) ---

    private static (int, int)[][] BuildUnitsStatic()
    {
        var units = new List<(int, int)[]>();
        for (int r = 0; r < 9; r++)
            units.Add(Enumerable.Range(0, 9).Select(c => (r, c)).ToArray());
        for (int c = 0; c < 9; c++)
            units.Add(Enumerable.Range(0, 9).Select(r => (r, c)).ToArray());
        for (int br = 0; br < 3; br++)
            for (int bc = 0; bc < 3; bc++)
                units.Add(
                    Enumerable.Range(0, 3).SelectMany(r =>
                        Enumerable.Range(0, 3).Select(c => (br * 3 + r, bc * 3 + c))
                    ).ToArray()
                );
        return units.ToArray();
    }

    private static Dictionary<(int, int), List<(int, int)[]>> BuildCellUnitsStatic((int, int)[][] allUnits)
    {
        var cellUnits = new Dictionary<(int, int), List<(int, int)[]>>();
        for (int r = 0; r < 9; r++)
            for (int c = 0; c < 9; c++)
                cellUnits[(r, c)] = allUnits.Where(u => u.Contains((r, c))).ToList();
        return cellUnits;
    }

    private static Dictionary<(int, int), HashSet<(int, int)>> BuildPeersStatic(
        Dictionary<(int, int), List<(int, int)[]>> cellUnits)
    {
        var peers = new Dictionary<(int, int), HashSet<(int, int)>>();
        for (int r = 0; r < 9; r++)
            for (int c = 0; c < 9; c++)
            {
                var cell = (r, c);
                peers[cell] = new HashSet<(int, int)>(
                    cellUnits[cell].SelectMany(u => u).Where(p => p != cell)
                );
            }
        return peers;
    }
}

Console.WriteLine("Classe NorvigPropagation definie (elimination + naked single + MRV)");

Classe NorvigPropagation definie (elimination + naked single + MRV)


### Lecture de la définition : le moteur de propagation et son API

La sortie confirme la définition : `Classe NorvigPropagation definie (elimination + naked single + MRV)`. Lisez l'API publique comme la table d'instruments qu'elle est — chaque membre prépare un geste précis de la suite du notebook :

| Membre | Rôle |
|---|---|
| `Initialize(grid)` | Charge la grille et déroule la propagation jusqu'au point fixe ; renvoie `false` si une contradiction est détectée |
| `Assign` / `Eliminate` | Les deux règles de la section précédente, mutuellement déclenchantes : assigner un hidden single provoque des éliminations chez les voisins, qui peuvent créer de nouveaux singletons |
| `PropagationCount` | Compteur d'éliminations — la mesure quantitative des sections 3 et 4 |
| `CloneState` / `RestoreState` | Sauvegarde et restauration de l'état complet des candidats : le mécanisme **de backtracking** du solveur, embarqué dès le moteur |
| `GetMrvCell` | L'heuristique MRV fournie par le moteur lui-même : la cellule non résolue au domaine le plus petit |
| `IsSolved` | Le point d'arrêt : toutes les cellules réduites à un singleton |

Un détail d'implémentation mérite l'œil : `Units`, `CellUnits` et `Peers` sont déclarés `static readonly` — les 27 unités et leurs 20 voisins par cellule, vérifiés en section 2, sont construits **une seule fois** et partagés entre toutes les instances, quel que soit le nombre de grilles résolues. La cellule suivante met immédiatement cet instrument à l'épreuve sur une grille facile.

### Démonstration : la propagation seule face à une grille facile

Le protocole de ce test isole volontairement la propagation : on construit un `NorvigPropagation`, on appelle `Initialize` — qui déroule élimination et naked singles jusqu'au point fixe — et on n'appelle **jamais** la recherche récursive. Trois mesures en sortent : le nombre de cellules vides au départ, le compteur `PropagationCount`, et le verdict `IsSolved`. L'hypothèse à tester est celle de l'introduction : sur une grille suffisamment fournie en indices, la cascade de déductions doit refermer la grille complète — zéro recherche, zéro retour en arrière. Une contradiction (une cellule perdant son dernier candidat) ferait échouer `Initialize` sur-le-champ ; si la sortie affiche un succès, ce sera donc un succès de pure inférence.

In [6]:
// Test de la propagation seule sur un puzzle facile
var easyGrid = SudokuHelper.GetSudokus(SudokuDifficulty.Easy).First();
display($"Grille initiale :\n{easyGrid}");
display($"Cellules vides : {easyGrid.NbEmptyCells()}");

var propagation = new NorvigPropagation();
bool success = propagation.Initialize(easyGrid);

display($"Propagation reussie : {success}");
display($"Nombre d'eliminations effectuees : {propagation.PropagationCount}");
display($"Grille resolue par propagation seule : {propagation.IsSolved()}");

// Afficher le résultat
if (propagation.IsSolved())
{
    var result = (SudokuGrid)easyGrid.Clone();
    propagation.WriteTo(result);
    display($"Solution :\n{result}");
    display($"Nombre d'erreurs : {result.NbErrors(SudokuHelper.GetSudokus(SudokuDifficulty.Easy).First())}");
}

Grille initiale :
-------------------------------
| 9     2 |       5 | 4     3 | 
| 1       |    6  3 |    2  5 | 
| 5     8 | 4     7 |    6    | 
-------------------------------
|    2  6 | 3     9 |       1 | 
|    5  7 |    1    | 2  9    | 
|    9    | 6  7    | 5  3    | 
-------------------------------
| 2  4    | 5  3    | 6       | 
| 7     5 | 2       | 3     4 | 
|    8    |    4  1 | 9  5    | 
-------------------------------

Cellules vides : 36

Propagation reussie : True

Nombre d'eliminations effectuees : 648

Grille resolue par propagation seule : True

Solution :
-------------------------------
| 9  6  2 | 1  8  5 | 4  7  3 | 
| 1  7  4 | 9  6  3 | 8  2  5 | 
| 5  3  8 | 4  2  7 | 1  6  9 | 
-------------------------------
| 8  2  6 | 3  5  9 | 7  4  1 | 
| 3  5  7 | 8  1  4 | 2  9  6 | 
| 4  9  1 | 6  7  2 | 5  3  8 | 
-------------------------------
| 2  4  9 | 5  3  8 | 6  1  7 | 
| 7  1  5 | 2  9  6 | 3  8  4 | 
| 6  8  3 | 7  4  1 | 9  5  2 | 
-------------------------------

Nombre d'erreurs : 0

### Interpretation : la grille facile refermée par pure inférence

Le verdict est sans appel : sur cette grille à **36 cases vides**, la propagation accomplit **648 éliminations** et referme la grille entière — `Grille resolue par propagation seule : True` — et la solution complète s'affiche sous vos yeux, identique à celle qu'une recherche exhaustive aurait trouvée. Aucune recherche récursive n'a été appelée : chaque case s'est déduite de la précédente par la seule cascade élimination → naked single → nouvelle élimination.

C'est le point central de Norvig démontré sur le terrain : deux règles locales, appliquées jusqu'au point fixe, capturent toute la logique de cette grille. Notez le rapport des deux compteurs : 648 éliminations pour 36 cases refermées, soit en moyenne 18 retraits de candidats chez les voisins par valeur déduite — c'est cette *multiplication* qui fait de la propagation un mécanisme de raisonnement et non un simple filtrage. La question devient alors : où cela s'arrête-t-il ? Testons maintenant une grille difficile pour trouver la limite.

In [7]:
// Test de la propagation seule sur un puzzle difficile
var hardGrid = SudokuHelper.GetSudokus(SudokuDifficulty.Hard).First();
display($"Grille initiale (Hard) :\n{hardGrid}");
display($"Cellules vides : {hardGrid.NbEmptyCells()}");

var propHard = new NorvigPropagation();
bool successHard = propHard.Initialize(hardGrid);

display($"Propagation reussie (pas de contradiction) : {successHard}");
display($"Nombre d'eliminations effectuees : {propHard.PropagationCount}");
display($"Grille resolue par propagation seule : {propHard.IsSolved()}");

if (!propHard.IsSolved())
{
    // Compter les cellules restant a resoudre
    int unresolved = 0;
    int totalCandidates = 0;
    for (int r = 0; r < 9; r++)
        for (int c = 0; c < 9; c++)
        {
            int count = propHard.GetCandidates(r, c).Count;
            if (count > 1)
            {
                unresolved++;
                totalCandidates += count;
            }
        }
    display($"Cellules non resolues apres propagation : {unresolved}");
    display($"Nombre moyen de candidats par cellule non resolue : {(double)totalCandidates / unresolved:F1}");
    display("La recherche recursive sera necessaire pour completer la resolution.");
}

Grille initiale (Hard) :
-------------------------------
| 4       |         | 8     5 | 
|    3    |         |         | 
|         | 7       |         | 
-------------------------------
|    2    |         |    6    | 
|         |    8    | 4       | 
|         |    1    |         | 
-------------------------------
|         | 6     3 |    7    | 
| 5       | 2       |         | 
| 1     4 |         |         | 
-------------------------------

Cellules vides : 64

Propagation reussie (pas de contradiction) : True

Nombre d'eliminations effectuees : 438

Grille resolue par propagation seule : False

Cellules non resolues apres propagation : 61

Nombre moyen de candidats par cellule non resolue : 4,4

La recherche recursive sera necessaire pour completer la resolution.

### Interpretation : où la propagation cale

Le contraste avec la grille facile est net. Sur cette grille difficile à **64 cases vides**, la propagation accomplit **438 éliminations** puis s'arrête : **61 cases restent non résolues**, chacune gardant en moyenne **4,4 candidats**. Aucune contradiction n'est survenue (`Propagation reussie : True`) — la propagation n'a pas *échoué*, elle a atteint son **point fixe** : plus aucun naked single n'est détectable, plus aucune élimination n'est déclenchable. L'information restante exige de trancher entre hypothèses concurrentes — exactement le travail d'une recherche.

| Grille testée | Effet mesuré de la propagation seule | Recherche nécessaire |
|---|---|---|
| Facile (36 vides) | Referme la grille : 648 éliminations, 36/36 cases | Non |
| Difficile (64 vides) | Réduit sans refermer : 438 éliminations, 61 cases ouvertes à 4,4 candidats | Oui |

La marge est étroite entre les deux colonnes du tableau : 4,4 candidats par cellule, c'est un facteur de branchement d'environ 4 (contre 9 sans aucune propagation) — la recherche récursive qui prend le relais en section 4 part d'un espace par case déjà réduit de moitié, et d'un espace total réduit bien davantage. C'est exactement ce que Norvig a observé : la propagation fait le gros du travail, et la recherche ne fait que trancher les derniers points d'indécision.

## Exercice : Detection de paires nues (Naked Pairs)

**Objectif :**
Implementez la detection des Naked Pairs : quand deux cellules d'une même unité
ont exactement les mêmes deux candidats, ces valeurs peuvent etre eliminees
des autres cellules de l'unité.

**Indice :**
Parcourez chaque unité et cherchez des paires de cellules avec les mêmes 2 candidats.


In [8]:
// EXERCICE : Detection de paires nues (Naked Pairs)
public List<((int, int) Cell1, (int, int) Cell2, HashSet<int> Values)> FindNakedPairs(Dictionary<(int, int), HashSet<int>> candidates)
{
    // TODO: Trouvez les paires nues dans le dictionnaire de candidats
    // Retournez la liste des paires trouvees avec leurs positions et valeurs
    return null; // TODO etudiant
}
Console.WriteLine("Exercice a completer");

Exercice a completer


## 4. Recherche récursive avec MRV

Lorsque la propagation ne suffit pas, on complète la résolution par une recherche récursive :

1. **Choisir la cellule** avec le moins de candidats restants (heuristique MRV -- Minimum Remaining Values). En choisissant la cellule la plus contrainte, on réduit au maximum le facteur de branchement.
2. **Essayer chaque candidat** : pour chaque valeur possible, on sauvegarde l'état, on assigne la valeur (ce qui déclenche une nouvelle propagation), et on recurse.
3. **Backtracking** : si une contradiction est detectee (la propagation renvoie `false`), on restaure l'état et on essaie le candidat suivant.
4. **Terminaison** : si toutes les cellules sont resolues, on a trouvé la solution.

Cette combinaison propagation + recherche MRV est extrêmement efficace : la propagation élague massivement l'arbre de recherche, et l'heuristique MRV oriente la recherche vers les branches les plus prometteuses.

### Implémentation du solveur complet


In [9]:
/// <summary>
/// Solveur de Sudoku base sur l'approche de Peter Norvig :
/// propagation de contraintes (élimination + naked singles) + recherche recursive MRV.
/// </summary>
public class NorvigSolver : ISudokuSolver
{
    // Compteurs de diagnostic
    public int SearchCalls { get; private set; }
    public int PropagationOnlySolved { get; private set; }

    public SudokuGrid Solve(SudokuGrid s)
    {
        SearchCalls = 0;
        PropagationOnlySolved = 0;

        var result = (SudokuGrid)s.Clone();
        var propagation = new NorvigPropagation();

        // Phase 1 : initialisation + propagation
        if (!propagation.Initialize(result))
            throw new InvalidOperationException("Grille invalide : contradiction detectee lors de la propagation initiale.");

        // Phase 2 : la propagation a-t-elle suffi ?
        if (propagation.IsSolved())
        {
            PropagationOnlySolved = 1;
            propagation.WriteTo(result);
            return result;
        }

        // Phase 3 : recherche recursive
        if (Search(propagation))
        {
            propagation.WriteTo(result);
            return result;
        }

        throw new InvalidOperationException("Grille sans solution.");
    }

    /// <summary>
    /// Recherche recursive avec heuristique MRV et backtracking.
    /// </summary>
    private bool Search(NorvigPropagation propagation)
    {
        SearchCalls++;

        // Vérifier si la grille est resolue
        if (propagation.IsSolved())
            return true;

        // Choisir la cellule avec le moins de candidats (MRV)
        var mrvCell = propagation.GetMrvCell();
        if (mrvCell == null)
            return false; // Pas de cellule non resolue (ne devrait pas arriver ici)

        var (row, col) = mrvCell.Value;
        var candidates = new List<int>(propagation.GetCandidates(row, col));

        foreach (var value in candidates)
        {
            // Sauvegarder l'etat avant l'essai
            var savedState = propagation.CloneState();

            // Essayer d'assigner cette valeur
            if (propagation.Assign(row, col, value))
            {
                // La propagation n'a pas detecte de contradiction
                if (Search(propagation))
                    return true; // Solution trouvee
            }

            // Backtrack : restaurer l'etat
            propagation.RestoreState(savedState);
        }

        return false; // Aucun candidat ne fonctionne
    }
}

Console.WriteLine("Classe NorvigSolver definie (propagation + recherche recursive MRV)");

Classe NorvigSolver definie (propagation + recherche recursive MRV)


### Lecture de la définition : le solveur complet et ses deux compteurs

La sortie confirme : `Classe NorvigSolver definie (propagation + recherche recursive MRV)`. La structure est un dialogue entre deux méthodes — `Solve` appelle d'abord `Initialize` sur un `NorvigPropagation` frais (toute la puissance de la section 3), puis délègue à `Search` si la grille n'est pas refermée. `Search` manipule le moteur, pas une grille de chiffres : il choisit la cellule à brancher via `GetMrvCell`, sauvegarde l'état des candidats par `CloneState`, assigne un candidat — ce qui redéclenche la propagation — et en cas de contradiction restaure tout par `RestoreState` avant d'essayer le candidat suivant. Là où le backtracking de Sudoku-01 revenait sur des cases une à une, ici le retour arrière restaure un **instantané entier** du raisonnement : on défait d'un coup tout ce qu'une hypothèse avait déduit.

Les deux compteurs publics sont les instruments de mesure de la section qui suit : `SearchCalls` comptera les plongées dans la recherche (0 si la propagation a suffi), et `PropagationOnlySolved` vaudra 1 lorsque la grille aura été refermée sans aucun appel. Ce sont eux qui transformeront la prochaine sortie en donnée exploitable plutôt qu'en impression.

### Test du solveur complet sur les trois grilles de référence

Le protocole réutilise les trois grilles affichées à l'ouverture. `SudokuHelper.SolveSudoku` mesure le temps et vérifie la validité de chaque solution ; les compteurs du solveur donnent la profondeur de recherche réellement nécessaire. La lecture se fait sur la ligne `Appels de recherche, Resolu par propagation seule` de chaque puzzle : la grille facile devrait être refermée par la propagation seule (0 appel), et les grilles plus dures activer la recherche — la question est *combien d'appels*. L'ordre de grandeur à garder en tête vient de Sudoku-01 : sur ces mêmes grilles, le backtracking naïf y affichait 122, 490 304 puis 12 625 368 appels de recherche. Si la propagation tient sa promesse, les compteurs de ce solveur devraient rester dans un autre monde.

In [10]:
var norvigSolver = new NorvigSolver();

// Test sur un puzzle facile
var easy = SudokuHelper.GetSudokus(SudokuDifficulty.Easy).First();
Console.WriteLine("=== Puzzle Facile ===");
SudokuHelper.SolveSudoku(easy, norvigSolver);
display($"Appels de recherche : {norvigSolver.SearchCalls}, Resolu par propagation seule : {norvigSolver.PropagationOnlySolved == 1}");

// Test sur un puzzle moyen
var medium = SudokuHelper.GetSudokus(SudokuDifficulty.Medium).First();
Console.WriteLine("\n=== Puzzle Moyen ===");
SudokuHelper.SolveSudoku(medium, norvigSolver);
display($"Appels de recherche : {norvigSolver.SearchCalls}, Resolu par propagation seule : {norvigSolver.PropagationOnlySolved == 1}");

// Test sur un puzzle difficile
var hard = SudokuHelper.GetSudokus(SudokuDifficulty.Hard).First();
Console.WriteLine("\n=== Puzzle Difficile ===");
SudokuHelper.SolveSudoku(hard, norvigSolver);
display($"Appels de recherche : {norvigSolver.SearchCalls}, Resolu par propagation seule : {norvigSolver.PropagationOnlySolved == 1}");

=== Puzzle Facile ===


Résolution par le solver NorvigSolver du Sudoku:
 -------------------------------
| 9     2 |       5 | 4     3 | 
| 1       |    6  3 |    2  5 | 
| 5     8 | 4     7 |    6    | 
-------------------------------
|    2  6 | 3     9 |       1 | 
|    5  7 |    1    | 2  9    | 
|    9    | 6  7    | 5  3    | 
-------------------------------
| 2  4    | 5  3    | 6       | 
| 7     5 | 2       | 3     4 | 
|    8    |    4  1 | 9  5    | 
-------------------------------

Sudoku renvoyé:
-------------------------------
| 9  6  2 | 1  8  5 | 4  7  3 | 
| 1  7  4 | 9  6  3 | 8  2  5 | 
| 5  3  8 | 4  2  7 | 1  6  9 | 
-------------------------------
| 8  2  6 | 3  5  9 | 7  4  1 | 
| 3  5  7 | 8  1  4 | 2  9  6 | 
| 4  9  1 | 6  7  2 | 5  3  8 | 
-------------------------------
| 2  4  9 | 5  3  8 | 6  1  7 | 
| 7  1  5 | 2  9  6 | 3  8  4 | 
| 6  8  3 | 7  4  1 | 9  5  2 | 
-------------------------------
Nombre d'erreurs réstantes: 0
Temps de résolution: 3,2103 ms

Appels de recherche : 0, Resolu par propagation seule : True


=== Puzzle Moyen ===


Résolution par le solver NorvigSolver du Sudoku:
 -------------------------------
| 8  5    |       2 | 4       | 
| 7  2    |         |       9 | 
|       4 |         |         | 
-------------------------------
|         | 1     7 |       2 | 
| 3     5 |         | 9       | 
|    4    |         |         | 
-------------------------------
|         |    8    |    7    | 
|    1  7 |         |         | 
|         |    3  6 |    4    | 
-------------------------------

... (2 outputs left out)


=== Puzzle Difficile ===


Résolution par le solver NorvigSolver du Sudoku:
 -------------------------------
| 4       |         | 8     5 | 
|    3    |         |         | 
|         | 7       |         | 
-------------------------------
|    2    |         |    6    | 
|         |    8    | 4       | 
|         |    1    |         | 
-------------------------------
|         | 6     3 |    7    | 
| 5       | 2       |         | 
| 1     4 |         |         | 
-------------------------------

Sudoku renvoyé:
-------------------------------
| 4  1  7 | 3  6  9 | 8  2  5 | 
| 6  3  2 | 1  5  8 | 9  4  7 | 
| 9  5  8 | 7  2  4 | 3  1  6 | 
-------------------------------
| 8  2  5 | 4  3  7 | 1  6  9 | 
| 7  9  1 | 5  8  6 | 4  3  2 | 
| 3  4  6 | 9  1  2 | 7  5  8 | 
-------------------------------
| 2  8  9 | 6  4  3 | 5  7  1 | 
| 5  7  3 | 2  9  1 | 6  8  4 | 
| 1  6  4 | 8  7  5 | 2  9  3 | 
-------------------------------
Nombre d'erreurs réstantes: 0
Temps de résolution: 6,1534 ms

Appels de recherche : 16, Resolu par propagation seule : False

### Interpretation : trois grilles, trois profondeurs de recherche

Les trois compteurs racontent une gradation parfaitement lisible : **0 appel** sur la facile — la propagation seule a refermé (`Resolu par propagation seule : True`) — puis **5 appels** sur la moyenne et **16 appels** sur la difficile. La recherche ne fait que trancher les derniers points d'indécision laissés par la propagation : même sur la grille difficile, seize plongées suffisent.

La référence rend la mesure saisissante : sur ces trois grilles **exactes** (mêmes fichiers, affichés à l'ouverture), le backtracking naïf de Sudoku-01 affichait 122 / 490 304 / **12 625 368** appels de recherche. Sur la grille difficile, le rapport n'est pas un facteur d'amélioration, c'est un changement de nature : 16 contre 12,6 millions, près de six ordres de grandeur. C'est l'effet combiné de la propagation — qui élague des sous-arbres entiers avant qu'ils n'existent — et de MRV — qui branche là où l'arbre est le plus étroit.

Les temps individuels restent de l'ordre de la milliseconde à quelques millisecondes (3,2 / 8,1 / 6,2 ms sur cette session) : aucune de ces grilles ne constitue une menace pour le solveur complet.

## 5. Tests de performance et comparaison

La démonstration précédente portait sur trois grilles isolées ; le benchmark systématise : chaque solveur affronte **10 grilles par difficulté**, avec limite de temps par run. La comparaison porte sur :

- **Temps total** de résolution des 10 grilles de chaque difficulté ;
- **Taux de succès** : le nombre de grilles résolues dans le délai — un solveur qui dépasse la limite sur une difficulté y est **disqualifié** (`Disqualified` dans le tableau final), pas seulement ralenti.

Deux concurrents entrent en piste : le solveur Norvig, et le solveur de référence défini à la section suivante — un backtracking volontairement naïf, copie de celui de Sudoku-01.

In [11]:
// Définition du solveur de référence : backtracking simple (même implémentation que Sudoku-01)
public class BacktrackingSolver : ISudokuSolver
{
    public SudokuGrid Solve(SudokuGrid s)
    {
        var result = (SudokuGrid)s.Clone();
        Search(result, 0, 0);
        return result;
    }

    private bool Search(SudokuGrid s, int row, int col)
    {
        if (row == 9) return true;
        if (col == 9) return Search(s, row + 1, 0);
        if (s.Cells[row, col] != 0) return Search(s, row, col + 1);

        for (int num = 1; num <= 9; num++)
        {
            if (IsValid(s, row, col, num))
            {
                s.Cells[row, col] = num;
                if (Search(s, row, col + 1)) return true;
                s.Cells[row, col] = 0;
            }
        }
        return false;
    }

    private bool IsValid(SudokuGrid s, int row, int col, int val)
    {
        for (int i = 0; i < 9; i++)
            if (s.Cells[row, i] == val || s.Cells[i, col] == val)
                return false;
        int sr = 3 * (row / 3), sc = 3 * (col / 3);
        for (int i = 0; i < 3; i++)
            for (int j = 0; j < 3; j++)
                if (s.Cells[sr + i, sc + j] == val)
                    return false;
        return true;
    }
}

Console.WriteLine("Classe BacktrackingSolver definie (solveur de reference pour comparaison)");

Classe BacktrackingSolver definie (solveur de reference pour comparaison)


### Le rôle du solveur de référence

Ce `BacktrackingSolver` est le **témoin** de l'expérience : un backtracking volontairement naïf (première cellule libre, première valeur du domaine, aucune propagation), écrit pour être *identiquement simple* à l'approche Norvig. Sans témoin, le benchmark de la section 5 ne montrerait rien — comparer un solveur à lui-même n'a pas de sens. Avec lui, chaque ligne du tableau isolera exactement ce que la propagation apporte : c'est ce solveur qui sera **disqualifié** sur les grilles difficiles pendant que Norvig les résoudra toutes.

Lançons maintenant le benchmark : dix grilles par difficulté pour chaque solveur, temps total et statut mesurés par `SudokuHelper.TestSolvers`. Deux choses à guetter dans le tableau qui suit. D'abord la colonne `Statut` : le témoin backtracking devrait passer les grilles faciles, mais toute la question est sa tenue sur les difficiles — le Sudoku-01 laissait déjà présager un effondrement (8,4 s et 12,6 millions d'appels sur une seule grille difficile). Ensuite la colonne `Temps` : l'écart entre les deux solveurs devrait se creuser à mesure que la difficulté monte, car c'est là que la propagation élague le plus. Le tableau ne jugera pas de la vitesse seule mais de la **robustesse** : réussir 10/10 partout, contre se disqualifier quelque part.

## Exercice : Analyser l'impact de la propagation

**Objectif :**
Desactivez l'étape de propagation dans le solveur Norvig et comparez
les performances avec et sans propagation.

**Indice :**
Modifiez le solveur pour court-circuiter l'étape eliminate et ne garder que assign.


In [12]:
// EXERCICE : Analyser l'impact de la propagation
public Dictionary<string, double> CompareWithAndWithoutPropagation(List<int[,]> puzzles)
{
    // TODO: Lancez le solveur Norvig avec et sans propagation
    // et comparez les temps moyens de résolution
    return null; // TODO etudiant
}
Console.WriteLine("Exercice a completer");

Exercice a completer


In [13]:
// Benchmark comparatif
var solvers = new List<(string Name, ISudokuSolver Solver)>
{
    ("Backtracking Simple", new BacktrackingSolver()),
    ("Norvig (Propagation + MRV)", new NorvigSolver())
};

var results = SudokuHelper.TestSolvers(solvers);

// Affichage textuel des résultats
Console.WriteLine($"{"Solveur",-30} | {"Difficulte",-10} | {"Temps (ms)",-12} | {"Resolus",-8} | {"Statut",-12}");
Console.WriteLine(new string('-', 80));
foreach (var r in results)
{
    Console.WriteLine($"{r.SolverName,-30} | {r.Difficulty,-10} | {r.Time,-12:F1} | {r.SolvedCount,-8} | {r.Status,-12}");
}

Running tests...

Solveur                        | Difficulte | Temps (ms)   | Resolus  | Statut      


--------------------------------------------------------------------------------


Backtracking Simple            | Easy       | 150,0        | 10       | Success     


Backtracking Simple            | Medium     | 554,4        | 10       | Success     


Backtracking Simple            | Hard       | 3006,7       | 0        | Disqualified


Norvig (Propagation + MRV)     | Easy       | 42,0         | 10       | Success     


Norvig (Propagation + MRV)     | Medium     | 53,1         | 10       | Success     


Norvig (Propagation + MRV)     | Hard       | 282,6        | 10       | Success     


### Lecture du tableau : une disqualification et un écart qui se creuse

Deux lignes sautent aux yeux. **`Backtracking Simple | Hard | 3006,7 | 0 | Disqualified`** : sur les grilles difficiles, le backtracking naïf dépasse la limite de temps sans résoudre **un seul** des 10 puzzles — il est disqualifié, pas seulement lent. En face, Norvig résout les 10 grilles difficiles en **282,6 ms** au total : la propagation de contraintes a éliminé d'énormes sous-arbres avant même qu'ils n'existent.

L'autre lecture du tableau est la progression de l'écart : environ **3,6 fois** sur les grilles faciles (150,0 contre 42,0 ms), plus de **10 fois** sur les moyennes (554,4 contre 53,1 ms), puis disqualification pure et simple sur les difficiles. Plus la grille est contrainte, plus chaque tour de propagation déduit de valeurs d'un coup, et moins la recherche doit brancher — tandis que le backtracking naïf, lui, explore précisément ce que la propagation élimine. La difficulté *pour un humain* (indices rares, chaînes de déduction longues) n'est pas la difficulté *pour la propagation* : l'algorithme exploite le même mécanisme que le joueur expert — l'élimination massive — mais sans jamais se tromper de chemin.

Une précaution de lecture, enfin : ces temps sont ceux d'**une session** (kernel chaud pour les deux solveurs, mais effets JIT et GC compris) — comparez les ordres de grandeur et les verdicts (Success/Disqualified, 10/10 contre 0/10), pas les millisecondes au dixième près. Le verdict, lui, est robuste : quelle que soit la session, le backtracking naïf ne termine pas sur Hard.

### Le même verdict en image

`SudokuHelper.DisplayResults` traduit le tableau en graphique comparé. La lecture à faire est celle du tableau, mais d'un coup d'œil : comment l'écart entre les deux solveurs évolue-t-il avec la difficulté, et que devient la série difficile du témoin — du temps dépensé sans une seule grille résolue derrière (0/10) ? Une série de temps n'a de sens que rapportée à son taux de succès : c'est la différence entre une barre plus haute et une barre vide.

In [14]:
// Affichage graphique des résultats
SudokuHelper.DisplayResults(results);
Console.WriteLine("Graphique de comparaison des solveurs affiche");

Comparaison des solveurs - difficulte Easy (temps total, ms) 0 40.5 81 121.5 162 Backtracking Simple Norvig (Propagation + MRV)

Comparaison des solveurs - difficulte Medium (temps total, ms) 0 149.694 299.389 449.083 598.777 Backtracking Simple Norvig (Propagation + MRV)

Comparaison des solveurs - difficulte Hard (temps total, ms) 0 76.314 152.628 228.942 305.256 Norvig (Propagation + MRV)

Graphique de comparaison des solveurs affiche


### Interpretation : comparaison des performances

**Points clés, tous ancrés sur les sorties commises ci-dessus** :

1. **Grilles faciles** : Norvig termine déjà nettement plus vite — 42,0 ms contre 150,0 ms pour les 10 grilles (environ 3,6 fois) — la propagation refermant la grille testée sans aucun appel de recherche (0 appel, section 4).

2. **Grilles difficiles** : l'écart devient qualitatif — 282,6 ms et 10/10 pour Norvig, contre la disqualification du témoin (0/10, temps limite dépassé). La section 4 a mesuré **16 appels** de recherche sur la grille difficile, quand le backtracking naïf en demandait 12,6 millions sur la même (Sudoku-01) : la propagation élague l'arbre avant qu'il n'existe.

3. **Robustesse** : Norvig résout systématiquement toutes les grilles, y compris les plus difficiles, en restant sous 400 ms pour l'ensemble du corpus testé (42,0 + 53,1 + 282,6 ms).

| Critère | Backtracking simple | Norvig |
|---------|--------------------:|-------:|
| Appels de recherche — grille facile | 122 (Sudoku-01, même grille) | 0 (mesuré, section 4) |
| Appels de recherche — grille difficile | 12 625 368 (Sudoku-01, même grille) | 16 (mesuré, section 4) |
| Grilles difficiles résolues (benchmark) | 0/10 — Disqualified | 10/10 |
| Garantie de solution | Oui | Oui |
| Complexité implémentation | Faible | Moyenne |

> **Note** : pour une comparaison avec d'autres approches, voir les notebooks Sudoku-10 (OR-Tools), Sudoku-12 (Z3) et Sudoku-02 (Dancing Links).

## 6. Exemple guide

### Exemple guide 1 : Detection des paires nues (naked pairs)

La propagation de Norvig utilise deux stratégies (élimination + naked single). On peut l'ameliorer en ajoutant la detection des **paires nues** (naked pairs) :

> Si deux cellules d'une même unité ont exactement les mêmes deux candidats {a, b}, alors a et b peuvent etre retires des candidats de toutes les autres cellules de cette unité.

**A faire** : modifier la méthode `Eliminate` de `NorvigPropagation` pour ajouter cette troisième règle de propagation. Tester l'impact sur le nombre d'appels de recherche.

```csharp
// Indice : dans la méthode Eliminate, après la règle 2, ajouter :
// Règle 3 : naked pairs
// Pour chaque unité contenant la cellule :
//   Chercher les paires de cellules ayant exactement les mêmes 2 candidats
//   Si trouvee, eliminer ces 2 valeurs des autres cellules de l'unité
```

### Exemple guide 2 : Statistiques de propagation

Ecrire un programme qui parcourt les fichiers de puzzles et compte combien sont resolus par propagation seule (sans aucun appel de recherche).

**A faire** : completer le code ci-dessous.


In [15]:
// Exemple guide 2 : compter les puzzles resolus par propagation seule
// Completer le code ci-dessous

foreach (var difficulty in new[] { SudokuDifficulty.Easy, SudokuDifficulty.Medium, SudokuDifficulty.Hard })
{
    var puzzles = SudokuHelper.GetSudokus(difficulty);
    int totalPuzzles = puzzles.Count;
    int solvedByPropagationOnly = 0;

    foreach (var puzzle in puzzles)
    {
        var prop = new NorvigPropagation();
        prop.Initialize(puzzle);

        // TODO : vérifier si la propagation seule a suffi
        // if (...)
        //     solvedByPropagationOnly++;
    }

    display($"{difficulty} : {solvedByPropagationOnly}/{totalPuzzles} resolus par propagation seule");
}

Easy : 0/51 resolus par propagation seule

Medium : 0/11 resolus par propagation seule

Hard : 0/95 resolus par propagation seule

### Bilan de l'exercice : lire un squelette, pas une mesure

Précaution de lecture essentielle avant d'interpréter cette sortie : l'incrément du compteur vit derrière le `TODO` — la ligne `solvedByPropagationOnly++` est **commentée dans le squelette**, et la sortie affiche donc l'état initial du compteur, pas un résultat. Le `0/51, 0/11, 0/95` commis dit seulement que l'outillage parcourt bien les 157 grilles ; il ne dit **pas** que la propagation seule échoue partout — le carnet lui-même le réfute : la section 3 a montré la grille facile refermée par propagation seule, et la section 4 a mesuré `Resolu par propagation seule : True` sur cette même grille. Compléter l'exercice — incrémenter lorsque `IsSolved` renvoie vrai après `Initialize` — donnerait au minimum 1/51 sur Easy : la grille témoin de la section 3 est `GetSudokus(Easy).First()`, donc par construction membre du lot de 51 grilles que cette boucle parcourt.

La question posée à l'étudiant est précisément celle-là : **quelle fraction du corpus la propagation seule referme-t-elle ?** Ce carnet a mesuré les deux bornes du phénomène — une grille facile refermée sans recherche (section 3) et une grille difficile laissée à 61 cases ouvertes à 4,4 candidats — mais la distribution complète sur les 157 grilles est le travail de l'exercice, pas une donnée acquise.

La série dans son ensemble donne alors une triangulation du même phénomène : Sudoku-06 mesure la version « inférence maximale » (MAC : 79 ms, 81 assignations, 0 backtrack), Sudoku-11 la délègue à un solveur industriel (Choco via IKVM : solution trouvée en 727 ms), et ce notebook isole la mécanique « inférence + recherche » qui relie les deux. Trois architectures, une même leçon : la propagation transforme la nature du problème, elle ne le résout pas seule.

## Synthèse

Ce notebook a présenté l'approche de Peter Norvig pour la résolution de Sudoku, combinant propagation de contraintes et recherche récursive.

### Récapitulatif

| Composant | Rôle | Inspiration théorique |
|-----------|------|----------------------|
| Élimination | Retirer les valeurs assignées des voisins | Arc-consistance (CSP) |
| Hidden single | Assigner une valeur qui n'a qu'un emplacement dans une unité | Consistance de domaine |
| MRV | Choisir la cellule la plus contrainte pour la recherche | Heuristique de branchement |
| Backtracking | Explorer les branches en cas d'echec | Recherche en profondeur |

### Points clés du solveur

1. La **propagation de contraintes** est le mécanisme fondamental : elle referme la grille facile testée sans aucun appel de recherche (0 appel mesuré, section 4) et laisse la difficile à 61 cases ouvertes à 4,4 candidats — c'est elle qui fait le gros du travail
2. La **recherche récursive** n'intervient que comme complement, sur un arbre déjà fortement réduit : 16 appels sur la grille difficile
3. L'heuristique **MRV** garantit un facteur de branchement minimal au moment de trancher
4. Cette approche est un excellent exemple de la synergie entre **inference** (propagation) et **recherche** (backtracking) — mesurée ici à grille identique : 16 appels contre 12,6 millions pour le backtracking naïf de Sudoku-01, et 10/10 grilles résolues à toutes difficultés contre une disqualification du témoin sur les difficiles

### Pour aller plus loin

- **Techniques de propagation avancées** : paires nues, triples nus, X-wing, swordfish (voir Exercice 1)
- **Théorie de la propagation** : voir [CSP-2-Consistance](../Search/Part2-CSP/CSP-2-Consistency.ipynb) pour une présentation formelle de l'arc-consistance et des algorithmes AC-3, MAC
- **Comparaison avec les solveurs spécialisés** : OR-Tools (Sudoku-10) et Dancing Links (Sudoku-02) utilisent des mécanismes différents mais reposent aussi sur la propagation de contraintes

### Ressources

- [Peter Norvig - Solving Every Sudoku Puzzle (2006)](http://norvig.com/sudoku.html)
- [Repository de référence : Sudoku.Norvig](https://github.com/jsboigeEpita/2024-EPITA-SCIA-PPC-Sudoku-NLP)
- Russell & Norvig, *Artificial Intelligence: A Modern Approach*, Chapitre 6 (CSP)

## À retenir

Le portage C# de l'approche de **Peter Norvig** conserve les deux piliers — **propagation de contraintes** (`assign` / `eliminate` en cascade) puis **recherche MRV** — et le banc d'essai de la section 5 les sépare proprement du backtracking naïf :

- **Backtracking simple** : 150,0 ms de moyenne sur 10 grilles faciles, 554,4 ms sur les moyennes, et **disqualifié sur les difficiles** (0/10 résolu, 3 006,7 ms de moyenne avant abandon) — l'exploration aveugle explose.
- **Norvig (propagation + MRV)** : **10/10 aux trois niveaux** — 42,0 ms (facile), 53,1 ms (moyen), 282,6 ms (difficile) — jamais disqualifié.
- La propagation fait l'essentiel du travail : c'est elle qui referme les grilles faciles, la recherche MRV n'intervenant que sur ce qui résiste.

## Vérifiez votre compréhension

**1. Pourquoi le backtracking simple est-il « disqualifié » sur les grilles difficiles, et pas simplement « lent » ?**

<details><summary>Réponse</summary>

La ligne du banc d'essai rend **0 résolu sur 10** avec 3 006,7 ms de moyenne : le solveur dépasse le critère de qualification (temps limite) sur chacune des grilles difficiles. Sans heuristique, l'ordre d'exploration fixe fait dérouler d'immenses sous-arbres stériles avant la bonne branche.

</details>

**2. Que choisit MRV, et pourquoi cela réduit-il l'arbre de recherche ?**

<details><summary>Réponse</summary>

MRV (*Minimum Remaining Values*) choisit la case au **domaine le plus petit** pour brancher. Chaque décision élimine vite des branches entières : 63 à 14 retours-arrière sur les grilles faciles du carnet voisin, contre 4 201 pour l'ordre fixe.

</details>

**3. Quelle est la différence avec la formulation CSP générique de Sudoku-06 ?**

<details><summary>Réponse</summary>

Même fond (arc-consistance, propagation), styles opposés : Sudoku-06 instancie un cadre CSP complet (MRV, forward checking), tandis que Norvig écrit la propagation à la main, cellule par cellule, via `assign` et `eliminate` qui s'appellent mutuellement. Voir [Sudoku-06-AIMA-CSP](Sudoku-06-AIMA-CSP-CSharp.ipynb).

</details>